# Parte C — Modelo preditivo de risco com microdados da PNS 2019

**TCC — Piettro Pevidor Nakashoji · Ciência de Dados · IESB · 2026**

Re-treino do classificador de risco de diabetes usando o **maior conjunto de dados individuais brasileiros**
disponível: os microdados da **Pesquisa Nacional de Saúde (PNS 2019, IBGE/Fiocruz)** — ~280 mil registros.
Complementa o *benchmark* Pima com dado nacional e **compara três modelos** (Regressão Logística, Random
Forest e XGBoost) com **validação cruzada estratificada**.

### Por que a PNS (e não os dados agregados do painel)
O classificador prevê o risco **individual** — exige uma linha por pessoa, com fatores de risco *e* desfecho.
As séries do painel são **agregadas** (taxas), e treinar um classificador individual com elas seria
**falácia ecológica**. A PNS tem o dado no nível certo.

### Pré-requisitos
1. Baixe os microdados da PNS 2019 no FTP do IBGE (arquivo de dados de largura fixa `PNS_2019.txt`
   e o dicionário `dicionario_PNS_microdados_2019.xls`).
2. Coloque os dois em `reproducao/dados_brutos/` (pasta ignorada pelo git; os microdados não são versionados).
3. `pip install -r reproducao/requirements.txt` e execute o notebook a partir da pasta `reproducao/`.

> Os **códigos das variáveis** estão centralizados em `CONFIG` (célula abaixo). Confira-os no dicionário da
> sua versão — pequenas diferenças de nomenclatura entre edições são comuns (especialmente a variável de peso).

## 0. Setup e configuração

In [ ]:
import pandas as pd, numpy as np, json, pathlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, classification_report

RAW = pathlib.Path('dados_brutos')

# === Códigos das variáveis na PNS 2019 (confirme no dicionário oficial) ===
CONFIG = {
    'arquivo_dados':      RAW / 'PNS_2019.txt',
    'arquivo_dicionario': RAW / 'dicionario_PNS_microdados_2019.xls',
    'var_dm':   'Q03001',   # diagnóstico de diabetes (1=Sim, 2=Não)
    'var_hip':  'Q00201',   # diagnóstico de hipertensão (1=Sim)
    'var_sexo': 'C006',     # sexo (1=Homem, 2=Mulher)
    'var_idade':'C008',     # idade em anos
    'var_peso_kg':  'W00103',  # peso aferido (kg) - módulo antropometria
    'var_alt_cm':   'W00203',  # altura aferida (cm)
    'var_peso_amostral': 'V0029',  # peso amostral do morador selecionado (confirme!)
}
print('Config pronta.')

## 1. Ler o dicionário e montar os `colspecs`

O microdado é de **largura fixa**: cada variável ocupa posições específicas de cada linha. Em vez de
digitar posições à mão, lemos o **dicionário oficial** e construímos automaticamente os `colspecs`
apenas para as variáveis de interesse.

In [ ]:
def montar_colspecs(dic_path, variaveis):
    """Lê o dicionário (.xls) e devolve (colspecs, names) para read_fwf, só das variáveis pedidas."""
    dic = pd.read_excel(dic_path, header=None)
    # localiza as colunas de posição inicial, tamanho e código (nomes variam por edição)
    txt = dic.astype(str)
    def acha_col(palavras):
        for j in range(dic.shape[1]):
            col = ' '.join(txt[j].head(8).tolist()).lower()
            if any(p in col for p in palavras):
                return j
        return None
    c_pos = acha_col(['posi', 'inicial'])
    c_tam = acha_col(['tamanho', 'tam'])
    c_cod = acha_col(['código', 'codigo', 'variável', 'variavel'])
    assert None not in (c_pos, c_tam, c_cod), 'Ajuste manual: não localizei as colunas no dicionário.'
    colspecs, names = [], []
    for _, row in dic.iterrows():
        cod = str(row[c_cod]).strip()
        if cod in variaveis:
            try:
                ini = int(float(row[c_pos])); tam = int(float(row[c_tam]))
            except (ValueError, TypeError):
                continue
            colspecs.append((ini - 1, ini - 1 + tam))  # read_fwf usa base 0
            names.append(cod)
    return colspecs, names

vars_uso = [CONFIG[k] for k in ['var_dm','var_hip','var_sexo','var_idade','var_peso_kg','var_alt_cm','var_peso_amostral']]
colspecs, names = montar_colspecs(CONFIG['arquivo_dicionario'], vars_uso)
print('Variáveis localizadas:', names)

## 2. Ler os microdados (apenas as colunas necessárias)

In [ ]:
df = pd.read_fwf(CONFIG['arquivo_dados'], colspecs=colspecs, names=names)
print('Dimensões brutas:', df.shape)
# o módulo de doenças crônicas (Q) é respondido pelo MORADOR SELECIONADO;
# linhas sem resposta de diabetes não são do morador selecionado -> descartar
df = df[df[CONFIG['var_dm']].isin([1, 2])].copy()
print('Após filtrar morador selecionado:', df.shape)

## 3. Engenharia de atributos

Construímos um conjunto individual com preditores clínicos/demográficos análogos ao FINDRISC.

In [ ]:
d = pd.DataFrame()
d['idade']    = pd.to_numeric(df[CONFIG['var_idade']], errors='coerce')
d['sexo_fem'] = (pd.to_numeric(df[CONFIG['var_sexo']], errors='coerce') == 2).astype(int)
d['hipertensao'] = (pd.to_numeric(df[CONFIG['var_hip']], errors='coerce') == 1).astype(int)
peso = pd.to_numeric(df[CONFIG['var_peso_kg']], errors='coerce')
alt  = pd.to_numeric(df[CONFIG['var_alt_cm']], errors='coerce')
d['imc'] = peso / (alt / 100.0) ** 2
d['peso_amostral'] = pd.to_numeric(df[CONFIG['var_peso_amostral']], errors='coerce')
d['dm'] = (pd.to_numeric(df[CONFIG['var_dm']], errors='coerce') == 1).astype(int)  # alvo

# limpeza: adultos, IMC plausível, sem ausentes nas features
d = d[(d['idade'] >= 18) & d['imc'].between(12, 60)]
d = d.dropna(subset=['idade', 'sexo_fem', 'hipertensao', 'imc', 'dm'])
print('Conjunto final:', d.shape)
print('Prevalência amostral de DM: %.1f%%' % (100 * d['dm'].mean()))
d.head()

## 4. Separação treino/teste

Split **estratificado** (mantém a proporção de casos). Nota: a PNS tem desenho amostral complexo (estratos,
conglomerados, pesos). Para *predição* tratamos como i.i.d.; estimativas populacionais formais exigiriam
`statsmodels`/`samplics` (ou o pacote `survey` no R). O peso amostral é lido, mas **não** é usado no treinamento: os modelos são ajustados sem ponderação, e o desbalanceamento das classes é tratado por `class_weight='balanced'`.

In [ ]:
features = ['idade', 'sexo_fem', 'hipertensao', 'imc']
X = d[features]
y = d['dm']
w = d['peso_amostral'].fillna(1.0)
Xtr, Xte, ytr, yte, wtr, wte = train_test_split(X, y, w, test_size=0.3, stratify=y, random_state=42)
print('Treino:', Xtr.shape, '| Teste:', Xte.shape)

## 5. Treino e validação cruzada de 3 modelos

In [ ]:
from xgboost import XGBClassifier

modelos = {
    'Regressão Logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight='balanced')),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.9,
                                         eval_metric='logloss', random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)
resultados = {}
for nome, mod in modelos.items():
    auc_cv = cross_val_score(mod, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    mod.fit(Xtr, ytr)
    proba = mod.predict_proba(Xte)[:, 1]
    resultados[nome] = {
        'auc_cv_media': round(auc_cv.mean(), 3),
        'auc_cv_dp':    round(auc_cv.std(), 3),
        'auc_teste':    round(roc_auc_score(yte, proba), 3),
        'f1_teste':     round(f1_score(yte, (proba >= 0.5).astype(int)), 3),
    }
    print(f'{nome:22s} AUC_cv={resultados[nome]["auc_cv_media"]:.3f}±{resultados[nome]["auc_cv_dp"]:.3f}  AUC_teste={resultados[nome]["auc_teste"]:.3f}')

## 6. Comparação e curvas ROC

In [ ]:
tabela = pd.DataFrame(resultados).T
display(tabela)

plt.figure(figsize=(6, 5))
for nome, mod in modelos.items():
    proba = mod.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(yte, proba)
    plt.plot(fpr, tpr, label=f'{nome} (AUC={resultados[nome]["auc_teste"]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=.5, label='Aleatório')
plt.xlabel('1 - Especificidade (FPR)'); plt.ylabel('Sensibilidade (TPR)')
plt.title('Curvas ROC - comparação de modelos (PNS 2019)'); plt.legend(); plt.tight_layout(); plt.show()

## 7. Importância das variáveis

In [ ]:
rf = modelos['Random Forest']
imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print('Importância (Random Forest):')
print(imp.round(3))
imp.sort_values().plot.barh(title='Importância das variáveis (Random Forest)'); plt.tight_layout(); plt.show()

## 8. Exportação dos resultados

Grava `modelos_pns_resultados.json`. Esses números (reais, do seu treino) podem substituir/citar os do
painel e entrar diretamente no capítulo de Resultados do TCC.

In [ ]:
saida = {
    'fonte': 'PNS 2019 (IBGE/Fiocruz) - microdados',
    'n_amostra': int(len(d)),
    'prevalencia_amostral_pct': round(100 * float(d['dm'].mean()), 1),
    'features': features,
    'modelos': resultados,
    'importancia_rf': {k: round(float(v), 3) for k, v in imp.items()},
}
pathlib.Path('modelos_pns_resultados.json').write_text(json.dumps(saida, ensure_ascii=False, indent=2), encoding='utf-8')
print('Exportado: modelos_pns_resultados.json')
print(json.dumps(saida, ensure_ascii=False, indent=2))

## 9. Exportação no formato do painel (`pns_results.js`)

Acrescenta aos resultados da seção 8 os elementos exibidos na aba *Modelagem Preditiva* do painel e relatados no Capítulo 6 do TCC: a curva ROC de cada modelo numa grade comum de taxas de falsos positivos (0 a 1, passo 0,025), a matriz de confusão da regressão logística no limiar de 0,5 e as razões de chances por desvio-padrão (exponencial dos coeficientes da regressão logística sobre as variáveis padronizadas).

Grava `pns_results.js`, que substitui o arquivo de mesmo nome na raiz do repositório.

In [ ]:
from sklearn.metrics import confusion_matrix

grade = np.round(np.linspace(0, 1, 41), 3)   # taxa de falsos positivos, passo 0,025
roc = {}
for nome, mod in modelos.items():
    fpr, tpr, _ = roc_curve(yte, mod.predict_proba(Xte)[:, 1])
    roc[nome] = np.round(np.interp(grade, fpr, tpr), 3).tolist()

principal = 'Regressão Logística'
lr = modelos[principal]
pred = (lr.predict_proba(Xte)[:, 1] >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
coef = lr.named_steps['logisticregression'].coef_[0]   # variáveis padronizadas => OR por 1 DP

painel = {
    'fonte': 'PNS 2019 (IBGE/Fiocruz) — microdados reais (subamostra de antropometria)',
    'n_amostra': saida['n_amostra'],
    'n_teste': int(len(yte)),
    'prevalencia_amostral_pct': saida['prevalencia_amostral_pct'],
    'features': features,
    'modelos': resultados,
    'importancia_rf': saida['importancia_rf'],
    'roc_grid': grade.tolist(),
    'roc': roc,
    'modelo_principal': principal,
    'matriz_confusao': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'coef_odds_por_dp': {f: round(float(np.exp(c)), 2) for f, c in zip(features, coef)},
}
pathlib.Path('pns_results.js').write_text('window.PNS_RESULTS = ' + json.dumps(painel, ensure_ascii=False) + ';',
                                         encoding='utf-8')
print('Exportado: pns_results.js')
print('Matriz de confusão (regressão logística, limiar 0,5):', painel['matriz_confusao'])
print('Razões de chances por DP:', painel['coef_odds_por_dp'])

## Conclusão e ressalvas

- **Re-treino em dado nacional individual** (PNS 2019) — supera a limitação do *benchmark* Pima e dá ao TCC
  resultados representativos da população brasileira.
- **Comparação justa** de 3 famílias de modelos com **validação cruzada estratificada** (k=5): a logística
  oferece interpretabilidade; árvores (RF/XGBoost) capturam não-linearidades e interações.
- **Ressalvas honestas:** (1) diagnóstico **autorreferido** subestima casos não diagnosticados; (2) o
  desenho amostral complexo da PNS (estratos/conglomerados) foi simplificado para a tarefa preditiva —
  estimativas populacionais formais usariam pacotes de *survey*; (3) classes desbalanceadas tratadas com
  `class_weight='balanced'` — pode-se ainda explorar SMOTE e ajuste de limiar priorizando **recall** (crítico em saúde).